# Module 5, Topic 5 — Building a Simple Semantic Search Engine

**Generative AI Fellowship — Beginner**

In this notebook, we assemble everything from this week — embeddings, similarity search, and a vector database — into one working semantic search engine over a small Lagos fashion store's FAQ page. Then we compare it directly against naive keyword search, on a real question a customer might actually type.

**What we'll do:**
1. Load a small FAQ document set
2. Train embeddings on a broader background corpus (not just the FAQs) so our model understands more everyday phrasing
3. Embed the FAQ documents and store them in ChromaDB
4. Accept a natural-language customer question and run semantic search
5. Run the identical question through a naive keyword search, built from scratch
6. Compare the two, side by side
7. Try a second question ourselves

> 💡 This notebook reuses every technique from Topics 2–4: training word vectors, averaging them into sentence vectors, and storing/querying them in ChromaDB with cosine similarity. Today is about wiring them together, not learning new mechanics.

## Step 1 — Install what we need

In [ ]:
!pip install chromadb gensim numpy --quiet

## Step 2 — Import what we need

In [ ]:
import numpy as np
import chromadb
from gensim.models import Word2Vec
from pprint import pprint

print("Ready to go!")

## Step 3 — Load our FAQ document set

Below are 20 FAQ entries for a small Lagos-based online fashion store, each tagged with a category — this is exactly the kind of small business use case described on Slide 10 of the Topic 5 deck.

In [ ]:
faqs = [
    {"id": "faq1",  "category": "delivery", "text": "orders within lagos are delivered within 2 to 3 business days"},
    {"id": "faq2",  "category": "delivery", "text": "orders outside lagos take 5 to 7 business days to arrive"},
    {"id": "faq3",  "category": "returns",  "text": "you can return an item within 7 days of delivery if unused and in original packaging"},
    {"id": "faq4",  "category": "returns",  "text": "refunds are processed within 5 business days after we receive your returned item"},
    {"id": "faq5",  "category": "product",  "text": "check our size chart on each product page before ordering"},
    {"id": "faq6",  "category": "returns",  "text": "we allow one free size exchange within 7 days of delivery"},
    {"id": "faq7",  "category": "payment",  "text": "we accept bank transfer debit card and cash on delivery"},
    {"id": "faq8",  "category": "payment",  "text": "cash on delivery is only available within lagos"},
    {"id": "faq9",  "category": "orders",   "text": "you will receive a tracking link via sms once your order ships"},
    {"id": "faq10", "category": "orders",   "text": "orders can be cancelled within 1 hour of placing them"},
    {"id": "faq11", "category": "payment",  "text": "enter your discount code at checkout to apply a reduction"},
    {"id": "faq12", "category": "product",  "text": "if an item is out of stock you can join the waitlist to be notified"},
    {"id": "faq13", "category": "support",  "text": "reach our support team via whatsapp or email for help"},
    {"id": "faq14", "category": "orders",   "text": "for bulk or wholesale orders please contact us directly"},
    {"id": "faq15", "category": "delivery", "text": "we currently do not ship outside nigeria"},
    {"id": "faq16", "category": "orders",   "text": "gift wrapping is available for an additional fee at checkout"},
    {"id": "faq17", "category": "product",  "text": "hand wash our ankara items in cold water to preserve the print"},
    {"id": "faq18", "category": "support",  "text": "we do not have a physical store all orders are online only"},
    {"id": "faq19", "category": "orders",   "text": "earn points on every purchase through our loyalty program"},
    {"id": "faq20", "category": "orders",   "text": "contact support immediately to change your delivery address before shipping"},
]

print(f"{len(faqs)} FAQ entries loaded, e.g.:")
pprint(faqs[6])

## Step 4 — A broader background corpus for training embeddings

Here's a real, practical problem: our 20 FAQs alone are too small and too narrow a corpus to teach a Word2Vec model that everyday words like "pay," "package," or "reach" relate to words actually used in our FAQs, like "transfer," "delivered," or "arrive."

This is the "pretrained vs. custom-trained" trade-off from Topic 2, Slide 12, in action: rather than training only on our tiny FAQ set, we add extra background sentences covering the same topics using more varied, everyday phrasing. We won't store these extra sentences as searchable documents — they only exist to help our embedding model learn broader word associations.

In [ ]:
background_sentences = [
    "my package will arrive within a few business days after it ships",
    "the courier delivers parcels within lagos every business day",
    "track your parcel until it arrives at your address",
    "your order arrives once it has been delivered to your address",
    "a delivered order usually arrives within a few business days",
    "customers can return a parcel if the item does not fit",
    "exchange your order for a different size or color if needed",
    "refund requests are reviewed after the returned item arrives",
    "pay with a debit card or transfer when you checkout",
    "contact an agent through whatsapp if you need help",
    "apply your discount code before you complete checkout",
    "join the waitlist if the item you want is out of stock",
]

print(f"{len(background_sentences)} background sentences added for training only")

## Step 5 — Train embeddings on FAQs + background corpus together

We train one Word2Vec model on the combined text — FAQs and background sentences — so it learns from both. When we build vectors for the FAQs in the next step, we'll use this richer model.

In [ ]:
training_sentences = [faq["text"].split() for faq in faqs] + [s.split() for s in background_sentences]

word_model = Word2Vec(
    sentences=training_sentences,
    vector_size=25,
    window=5,
    min_count=1,
    sg=1,
    epochs=400,
    seed=42,
    workers=1,
)

print("Vocabulary size:", len(word_model.wv.key_to_index))

## Step 6 — Turn each FAQ into a sentence vector

Same averaging approach as Topics 3 and 4 — with one refinement. Topic 3 noticed that common connecting words ("the," "is," "you," "can") dominate short averaged sentences and compress similarity scores together. Here, we filter those common words out before averaging, so the vector is built mostly from words that actually carry meaning.

In [ ]:
STOPWORDS = {
    "i", "a", "an", "the", "is", "are", "was", "were", "will", "when", "my", "your", "our",
    "you", "we", "to", "of", "in", "on", "for", "at", "and", "or", "if", "it", "its", "this",
    "that", "do", "does", "did", "can", "not", "no", "once", "after", "before", "within", "via", "with",
}


def sentence_vector(sentence, model):
    words = [w for w in sentence.split() if w not in STOPWORDS]
    word_vectors = [model.wv[w] for w in words if w in model.wv]
    if not word_vectors:
        # fall back to using every word if filtering removed everything
        word_vectors = [model.wv[w] for w in sentence.split() if w in model.wv]
    return np.mean(word_vectors, axis=0)


faq_embeddings = [sentence_vector(faq["text"], word_model) for faq in faqs]

print("Number of FAQ embeddings:", len(faq_embeddings))
print("Embedding shape:", faq_embeddings[0].shape)

## Step 7 — Store the FAQ embeddings in ChromaDB

Same setup as Topic 4: a cosine-distance collection, with the original FAQ text and category stored as metadata alongside each vector. Notice the background sentences from Step 4 are **not** stored here — they only helped train the embedding model, and aren't things a customer would ever want returned as a search result.

In [ ]:
chroma_client = chromadb.Client()

collection = chroma_client.create_collection(
    name="store_faqs",
    metadata={"hnsw:space": "cosine"},
)

collection.add(
    embeddings=[vector.tolist() for vector in faq_embeddings],
    documents=[faq["text"] for faq in faqs],
    metadatas=[{"category": faq["category"]} for faq in faqs],
    ids=[faq["id"] for faq in faqs],
)

print("FAQs stored in ChromaDB:", collection.count())

## Step 8 — Semantic search: a real customer question

Here's a question a customer might actually type — notice it doesn't use the exact wording of any FAQ (no "transfer," no "debit card," no "cash on delivery").

In [ ]:
customer_question = "can i pay with my phone"
question_vector = sentence_vector(customer_question, word_model)

semantic_results = collection.query(
    query_embeddings=[question_vector.tolist()],
    n_results=5,
)

print(f'Question: "{customer_question}"\n')
print("Top 5 matches (semantic search):")
for text, metadata, distance in zip(
    semantic_results["documents"][0],
    semantic_results["metadatas"][0],
    semantic_results["distances"][0],
):
    print(f"  distance={distance:.3f}  [{metadata['category']:<9}]  {text}")

**What to notice:** the top result should be the payment FAQ about accepting bank transfer, debit card, and cash on delivery — even though the customer's question shares **zero exact words** with it. This is the moment everything this week has been building toward: matching meaning, not matching text.

## Step 9 — The same question, with naive keyword search

Now let's build the simplest possible keyword search from scratch: for each FAQ, count how many of the customer's words appear in it, and rank by that count.

In [ ]:
def keyword_search(query, documents, top_n=5):
    query_words = set(w for w in query.split() if w not in STOPWORDS)
    scored = []
    for doc in documents:
        doc_words = set(w for w in doc["text"].split() if w not in STOPWORDS)
        overlap_count = len(query_words & doc_words)
        scored.append((overlap_count, doc))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return scored[:top_n]


keyword_results = keyword_search(customer_question, faqs)

print(f'Question: "{customer_question}"\n')
print("Top 5 matches (keyword search):")
for overlap_count, faq in keyword_results:
    print(f"  overlap={overlap_count}  [{faq['category']:<9}]  {faq['text']}")

**What to notice:** every single FAQ scores an overlap of `0` — the customer's words ("pay," "phone") never appear anywhere in our 20 FAQs. Naive keyword search has no signal to work with at all here, and just falls back to returning documents in whatever order they happened to be stored — completely disconnected from relevance.

## Step 10 — Semantic vs. keyword search, side by side

| | Semantic Search | Keyword Search |
|--|------------------|------------------|
| Top result | Correct payment FAQ | Arbitrary (all tied at 0) |
| Why | Learned that "pay" relates to "transfer / debit card / checkout" from the background corpus | Looked for exact word matches only, found none |
| Useful here? | Yes | No |

This is exactly the gap described on Topic 5, Slide 3 and Slide 10 — a real customer typing a completely reasonable, natural question would get nothing useful from keyword search on this FAQ page, but a correct answer from semantic search.

## Step 11 — Try it yourself

Change `customer_question` below to something else a customer might ask, and re-run both searches. A few ideas to try:
- `"how do i get a bigger size"` (compare against the size-exchange FAQ)
- `"my order got cancelled by accident"` (compare against the cancellation FAQ)
- `"do you have a shop i can visit"` (compare against the online-only FAQ)

For each one, check: did semantic search find something reasonable? Did keyword search find anything at all?

In [ ]:
customer_question_2 = "how do i get a bigger size"   # <-- try changing this

question_vector_2 = sentence_vector(customer_question_2, word_model)
semantic_results_2 = collection.query(query_embeddings=[question_vector_2.tolist()], n_results=3)

keyword_results_2 = keyword_search(customer_question_2, faqs, top_n=3)

print(f'Question: "{customer_question_2}"\n')

print("Semantic search:")
for text, metadata, distance in zip(
    semantic_results_2["documents"][0],
    semantic_results_2["metadatas"][0],
    semantic_results_2["distances"][0],
):
    print(f"  distance={distance:.3f}  [{metadata['category']:<9}]  {text}")

print("\nKeyword search:")
for overlap_count, faq in keyword_results_2:
    print(f"  overlap={overlap_count}  [{faq['category']:<9}]  {faq['text']}")

## Recap

In this notebook, we built a complete semantic search engine:
- Loaded a small business's FAQ document set
- Trained embeddings on a broader background corpus so our model understood more natural, everyday phrasing
- Stored embedded FAQs, with metadata, in a ChromaDB collection
- Answered a real customer question via semantic search — correctly, despite zero shared words
- Ran the same question through a naive keyword search, and watched it fail completely
- Compared the two side by side, and saw exactly why semantic search matters

This pipeline — documents → embeddings → vector store → query → ranked results — is also the **retrieval** half of Retrieval-Augmented Generation (RAG).

**Up next (Week 11):** instead of returning matching FAQ text for a customer to read themselves, we'll hand the retrieved chunks to an LLM and have it generate a direct, grounded answer.